In [ ]:
import subprocess, sys
for pkg in ['lightgbm', 'torch', 'scikit-learn', 'scipy', 'pandas', 'numpy', 'matplotlib']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print('✅ deps ok')

In [ ]:
import os, re, json, time, math, warnings
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve,
                             mean_absolute_error, mean_squared_error)
import lightgbm as lgb

import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Dark GitHub-style theme (как в исходном ноутбуке) ────────────────────────
plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9',
    'xtick.color':'#8b949e','ytick.color':'#8b949e',
    'text.color':'#c9d1d9','grid.color':'#21262d',
    'grid.linestyle':'--','grid.alpha':0.5,
    'font.family':'DejaVu Sans','axes.titlesize':12,'axes.labelsize':10,
    'legend.facecolor':'#161b22','legend.edgecolor':'#30363d',
})
PAL = ['#58a6ff','#3fb950','#ff7b72','#d2a8ff','#ffa657','#79c0ff']
SUB_PAL = {'559':'#58a6ff','563':'#3fb950','570':'#ff7b72',
           '575':'#d2a8ff','588':'#ffa657','591':'#79c0ff'}

print(f'✅ Imports done | LightGBM {lgb.__version__} | torch {torch.__version__} | device={DEVICE}')

In [ ]:
@dataclass
class Config:
    # ── Data ──────────────────────────────────────────────────────────────────
    DATA_DIR: str = '../kt3/data'      # папка с *-ws-training.xml / *-ws-testing.xml

    # ── Time-series parameters ────────────────────────────────────────────────
    FREQ_MIN: int    = 5
    WINDOW_MIN: int  = 60        # 60-min look-back  → 12 шагов истории
    HORIZON_MIN: int = 30        # 30-min точечный прогноз → 6 шагов (sanity-метрика)
    MIN_SEG_PTS: int = 48        # минимум 4-часовой непрерывный сегмент

    # ── Клиническая тревога ───────────────────────────────────────────────────
    HYPO_THRESHOLD: int   = 70   # mg/dL — порог гипогликемии
    HYPO_HORIZON_MIN: int = 60   # смотрим вперёд 60 мин: будет ли хоть одна точка < 70

cfg = Config()
FREQ = f'{cfg.FREQ_MIN}min'
W = cfg.WINDOW_MIN       // cfg.FREQ_MIN     # 12
H = cfg.HORIZON_MIN      // cfg.FREQ_MIN     # 6  (точечный +30 мин)
HYPO_STEPS = cfg.HYPO_HORIZON_MIN // cfg.FREQ_MIN  # 12 (тревога на 60 мин)

print('📋 Config:')
for k, v in vars(cfg).items(): print(f'   {k:20s} = {v}')
print(f'\n   Window steps (W)      = {W}')
print(f'   Point horizon (H)     = {H}  ({cfg.HORIZON_MIN} min)')
print(f'   Hypo horizon (steps)  = {HYPO_STEPS}  ({cfg.HYPO_HORIZON_MIN} min)')

DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), cfg.DATA_DIR))
if not os.path.isdir(DATA_DIR):
    alt = os.path.abspath(os.path.join(os.getcwd(), 'kt3', 'data'))
    if os.path.isdir(alt):
        DATA_DIR = alt
print(f'📂 Resolved DATA_DIR = {DATA_DIR}')

PATIENTS = ['559','563','570','575','588','591']
FMT      = "%d-%m-%Y %H:%M:%S"
STEP_MIN = 5
DT_H     = STEP_MIN/60.0    # 1 шаг = 5 мин; дрейф измеряем в «мг/дл·норм за час»
TAU_CARB, TAU_INS = 9, 11   # ~45/55 мин экспоненциальный спад «on-board» (в шагах)
torch.manual_seed(0); np.random.seed(0)

## 1. Парсинг → регулярная 5-минутная сетка, сегменты, экзогенные каналы

Главное: глюкоза кладётся на сетку 5 мин по округлённому индексу времени; разрывы становятся `NaN`. `contiguous_segments` отдаёт только непрерывные куски — внутри них корректно работает дискретизация SDE. `meal.carbs` и `bolus.dose` — это управляющие входы; превращаем их в импульсы и в экспоненциальные следы COB/IOB (carbs/insulin-on-board) — стандартный физиологический признак.

In [ ]:
def _ts(s):
    try: return datetime.strptime(s, FMT)
    except Exception: return None

def _events(root, tag, tkey='ts'):
    nodes = root.findall(tag)
    if not nodes:
        nodes = root.findall(f'.//{tag}')
    out=[]
    for node in nodes:
        for e in node.findall('.//event'):
            t=_ts(e.attrib.get(tkey,''))
            if t is not None: out.append((t, e.attrib))
    return out

def load_patient(path):
    import os
    # Resolve path: if file doesn't exist, try common data dirs and a repo search
    if not os.path.exists(path):
        basename = os.path.basename(path)
        candidates = [os.path.join('..','kt3','data',basename), os.path.join('kt3','data',basename)]
        found = None
        for c in candidates:
            if os.path.exists(c):
                found = c; break
        if found is None:
            for r,_,files in os.walk('.'):
                if basename in files:
                    found = os.path.join(r, basename); break
        if found is None:
            raise FileNotFoundError(f"File {path} not found; tried common locations and repository search")
        path = found

    # --- debugging prints to surface runtime state ---
    print(f"[debug] load_patient called with path: {path}")
    print(f"[debug] abs path: {os.path.abspath(path)}")

    root = ET.parse(path).getroot()
    tags = [child.tag for child in root]
    print(f"[debug] root.tag={root.tag}; root children={tags}")

    direct_nodes = root.findall('glucose_level')
    print(f"[debug] direct glucose_level nodes: {len(direct_nodes)}")
    for i,node in enumerate(direct_nodes[:3]):
        evs=node.findall('.//event')
        print(f"[debug]   node {i} events (direct): {len(evs)}; sample attrs: {evs[0].attrib if evs else None}")

    # now use the helper to collect events
    gl = sorted(_events(root,'glucose_level'), key=lambda x:x[0])
    print(f"[debug] _events returned {len(gl)} glucose-level event tuples")

    if len(gl) == 0:
        raise ValueError(f"No glucose_level events found in {path}. Available root children: {tags}")
    t0 = gl[0][0]
    gidx = lambda t: int(round((t-t0).total_seconds()/60.0/STEP_MIN))
    n = gidx(gl[-1][0])+1
    glu = np.full(n, np.nan)
    for t,a in gl: glu[gidx(t)] = float(a['value'])
    carb=np.zeros(n); ins=np.zeros(n)
    for t,a in _events(root,'meal'):
        i=gidx(t)
        if 0<=i<n and a.get('carbs','').strip(): carb[i]+=float(a['carbs'])
    for t,a in _events(root,'bolus',tkey='ts_begin'):
        i=gidx(t)
        if 0<=i<n and a.get('dose','').strip(): ins[i]+=float(a['dose'])
    return dict(glucose=glu, carb=carb, insulin=ins, n=n, pid=root.attrib.get('id'))

## 2. Признаки и нормировка

Состояние = z-нормированная глюкоза (статистики берём из train и переиспользуем на test). Вход дрейфа: `[x_norm, COB_norm, IOB_norm]`.

In [ ]:
def build_features(d, norm=None):
    glu=d['glucose'].copy()
    cob=iob_cob_traces(d['carb'],TAU_CARB); iob=iob_cob_traces(d['insulin'],TAU_INS)
    if norm is None:
        norm=dict(gm=np.nanmean(glu), gs=np.nanstd(glu),
                  cs=cob.std()+1e-6, is_=iob.std()+1e-6)
    xs=(glu-norm['gm'])/norm['gs']
    u=np.stack([cob/norm['cs'], iob/norm['is_']],axis=1)
    return xs,u,norm

def make_pairs(xs,u,segs):
    X0,U0,X1=[],[],[]
    for a,b in segs:
        seg=xs[a:b]
        if np.isnan(seg).any():            # интерполируем одиночные внутр. NaN
            idx=np.arange(len(seg)); g=~np.isnan(seg); seg=np.interp(idx,idx[g],seg[g])
        X0.append(seg[:-1]); X1.append(seg[1:]); U0.append(u[a:b-1])
    X0=np.concatenate(X0); X1=np.concatenate(X1); U0=np.concatenate(U0)
    t=lambda z: torch.tensor(z,dtype=torch.float32)
    return t(X0).unsqueeze(-1), t(U0), t(X1).unsqueeze(-1)

def iob_cob_traces(impulse, tau):
    out=np.zeros_like(impulse); decay=np.exp(-1.0/tau); acc=0.0
    for k in range(len(impulse)):
        acc = acc * decay + impulse[k]
        out[k] = acc
    return out

def contiguous_segments(glu, min_len=24):
    segs=[]; start=None
    for k in range(len(glu)):
        ok=not np.isnan(glu[k])
        if ok and start is None:
            start=k
        elif (not ok) and start is not None:
            if k-start>=min_len: segs.append((start,k))
            start=None
    if start is not None and len(glu)-start>=min_len:
        segs.append((start,len(glu)))
    return segs

## 3. Модель: нейросетевой дрейф + state-dependent диффузия

$$dX_t = f_\theta(X_t, u_t)\,dt + g_\theta(X_t, u_t)\,dW_t$$

Вход дрейфа и диффузии — `[состояние, COB, IOB]` (неавтономность). **Диффузия теперь обучаемая и зависит от состояния**: $g_\theta(X,u)$ — отдельная Tanh-MLP голова с `softplus` на выходе. Это принципиально: с константной $\sigma$ предсказательный интервал одинаков и при 200 mg/dL, и у порога гипо 70 — модель не может расширять неопределённость там, где она клинически важна. State-dependent $g_\theta$ — это то, чем Neural SDE структурно бьёт точечный baseline и ARIMA с фиксированной дисперсией.

In [ ]:
class DriftDiffNet(nn.Module):
    """Neural SDE с ОБУЧАЕМОЙ, state-dependent диффузией:
         dX = f_theta(X,u) dt + g_theta(X,u) dW
       Раньше sigma была скаляром (гомоскедастика) -> модель не могла расширять
       неопределённость у порога гипо / на крутом тренде. Теперь sigma = g_theta(X,u)."""
    def __init__(self, d_u=2, h=64, sigma_floor=1e-2):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(1+d_u, h), nn.Tanh(),
                               nn.Linear(h, h), nn.Tanh(), nn.Linear(h, 1))
        # голова диффузии — симметрично дрейфу, softplus гарантирует sigma>0
        self.g = nn.Sequential(nn.Linear(1+d_u, h), nn.Tanh(),
                               nn.Linear(h, h), nn.Tanh(), nn.Linear(h, 1))
        self.sigma_floor = sigma_floor

    def drift(self, x, u):
        return self.f(torch.cat([x, u], -1))

    def sigma_xu(self, x, u):
        """state-dependent диффузия g_theta(x,u); [.,1], строго > floor."""
        return torch.nn.functional.softplus(self.g(torch.cat([x, u], -1))) + self.sigma_floor

# Совместимость со старым кодом диагностики, где обращались к m.sigma как к скаляру:
# берём среднюю sigma при нулевых экзогенах вокруг нормального диапазона.
def _scalar_sigma(model):
    with torch.no_grad():
        xg = torch.linspace(-2, 2, 50).unsqueeze(-1)
        s = model.sigma_xu(xg, torch.zeros(50, 2)).mean().item()
    return s
DriftDiffNet.sigma = property(lambda self: torch.tensor(_scalar_sigma(self)))

# алиас, чтобы существующие вызовы DriftNet() продолжали работать
DriftNet = DriftDiffNet

## 4. Лосс: одношаговое псевдоправдоподобие Эйлера + Girsanov-KL

Данные — плотно наблюдаемая траектория диффузии, поэтому reconstruction-часть — псевдоправдоподобие Эйлера переходной плотности с **локальной** диффузией:
$$p(x_{k+1}\mid x_k,u_k)=\mathcal N\!\big(x_k+f_\theta(x_k,u_k)\,\Delta t,\ g_\theta(x_k,u_k)^2\,\Delta t\big).$$
Гетероскедастичный NLL сам учит $g_\theta$ через max-likelihood: где остатки систематически крупнее (крутой тренд, постпрандиальные скачки, близость к гипо) — модель локально поднимает $\sigma$. KL по Гирсанову против бездрейфового приора, теперь тоже с локальной диффузией:
$$\text{KL}\approx \frac{\Delta t}{2}\sum_k \frac{\|f_\theta(x_k,u_k)\|^2}{g_\theta(x_k,u_k)^2}.$$
При одношаговом MLE KL играет роль приора на величину дрейфа ($\beta$-регуляризация).

In [ ]:
def train(model, X0, U0, X1, n_iter=2500, bs=2048, beta_kl=1e-3, lr=3e-3, log=False):
    """Одношаговое Эйлерово псевдоправдоподобие.
       Δ-таргет (mean = x + f·dt) бьёт persistence-коллапс;
       state-dependent var = sigma(x,u)²·dt учит ГЕТЕРОСКЕДАСТИЧНОСТЬ через ML:
       большой остаток -> модель локально поднимает sigma."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_iter, 1e-4)
    N = X0.shape[0]
    for it in range(n_iter):
        idx = torch.randint(0, N, (bs,)); x, u, xn = X0[idx], U0[idx], X1[idx]
        f   = model.drift(x, u)
        sig = model.sigma_xu(x, u)                     # ЛОКАЛЬНАЯ диффузия
        mean = x + f * DT_H
        var  = sig**2 * DT_H
        nll = 0.5 * (((xn - mean)**2) / var + torch.log(var))      # гетероскед. NLL
        kl  = (f**2) / (2 * sig**2) * DT_H                          # Гирсанов с локальной sigma
        loss = nll.mean() + beta_kl * kl.mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if log and (it+1) % 500 == 0:
            print(f"  it{it+1} nll={nll.mean():.3f} sigma_mean={sig.mean().item():.3f}")
    return model

## 5. Прогноз: free-running сэмплинг траекторий + распределенческие метрики

Свободный прогон **с диффузией**: из каждой стартовой точки сэмплим `n_samples` траекторий (`X += f·dt + g·√dt·ξ`), ошибка копится — настоящий тест. **Честность:** будущие приёмы пищи/инсулин не объявлены, COB/IOB только спадают по экспоненте.

Из пучка траекторий считаем не только RMSE (по среднему), но и то, ради чего нужна SDE:
- **CRPS** — качество всего предсказательного распределения, а не моды;
- **coverage80** — доля попаданий факта в центральный 80% интервал (калибровка); цель ≈ 0.80;
- **P(hypo) → AUROC** — тревога «min глюкозы за 60 мин < 70».

Сравниваем с persistence (глюкоза = последнему значению) — стандартный барьер для CGM.

In [ ]:
@torch.no_grad()
def forecast_eval(model, xs, d, norm, segs, horizons=(6,12), n_samples=64,
                  hypo_threshold=70.0, hypo_steps=12):
    """Free-running оценка С ДИФФУЗИЕЙ: сэмплим n_samples траекторий и считаем
       не только RMSE (по среднему), но и распределенческие метрики:
         - CRPS(h)        — качество всего предсказательного распределения
         - coverage80(h)  — доля попаданий факта в центральный 80% интервал
         - P(hypo) / AUROC — тревога 'min глюкозы за горизонт < 70'
       Эти метрики и есть ниша Neural SDE; на константной sigma они не работали."""
    gm, gs = norm['gm'], norm['gs']
    cob = iob_cob_traces(d['carb'], TAU_CARB); iob = iob_cob_traces(d['insulin'], TAU_INS)
    Hmax = max(max(horizons), hypo_steps)
    sdt = DT_H ** 0.5

    errs = {h: [] for h in horizons}; per = {h: [] for h in horizons}
    crps = {h: [] for h in horizons}; cov = {h: [] for h in horizons}
    hypo_true, hypo_prob = [], []

    for a, b in segs:
        seg = xs[a:b]
        if np.isnan(seg).any(): continue
        for k in range(a, b - Hmax):
            j = k - a
            x0 = float(seg[j])
            # пучок траекторий из одной стартовой точки
            X = torch.full((n_samples, 1), x0)
            c = np.full(n_samples, cob[k], dtype=np.float64)
            i = np.full(n_samples, iob[k], dtype=np.float64)
            traj = np.empty((n_samples, Hmax), dtype=np.float64)
            for s in range(1, Hmax + 1):
                c *= np.exp(-1 / TAU_CARB); i *= np.exp(-1 / TAU_INS)
                uu = torch.tensor(np.stack([c / norm['cs'], i / norm['is_']], 1),
                                  dtype=torch.float32)
                f   = model.drift(X, uu)
                sig = model.sigma_xu(X, uu)
                X = X + f * DT_H + sig * sdt * torch.randn_like(X)
                traj[:, s-1] = X.squeeze(-1).numpy() * gs + gm     # денорм mg/dL
            truth_full = seg[j+1:j+1+Hmax] * gs + gm

            for h in horizons:
                samp_h = traj[:, h-1]                              # [n_samples]
                truth_h = float(truth_full[h-1])
                mean_h = samp_h.mean()
                errs[h].append((mean_h - truth_h)**2)
                per[h].append((x0 * gs + gm - truth_h)**2)
                # CRPS по сэмплам (энергетическая форма)
                t1 = np.abs(samp_h - truth_h).mean()
                t2 = np.abs(samp_h[:, None] - samp_h[None, :]).mean()
                crps[h].append(t1 - 0.5 * t2)
                # покрытие центрального 80%
                lo, hi = np.percentile(samp_h, [10, 90])
                cov[h].append(float(lo <= truth_h <= hi))

            # тревога: min глюкозы за hypo_steps
            p_hypo = float((traj[:, :hypo_steps].min(axis=1) < hypo_threshold).mean())
            y_hypo = float(truth_full[:hypo_steps].min() < hypo_threshold)
            hypo_prob.append(p_hypo); hypo_true.append(y_hypo)

    out = {}
    for h in horizons:
        out[h] = dict(
            rmse=float(np.sqrt(np.mean(errs[h]))),
            rmse_persist=float(np.sqrt(np.mean(per[h]))),
            crps=float(np.mean(crps[h])),
            coverage80=float(np.mean(cov[h])),
            n=len(errs[h]))
    # AUROC тревоги (если оба класса присутствуют)
    yt = np.array(hypo_true); pp = np.array(hypo_prob)
    auroc = float(roc_auc_score(yt, pp)) if (yt.min() != yt.max()) else float('nan')
    out['hypo'] = dict(auroc=auroc, prevalence=float(yt.mean()), n=len(yt))
    return out

## 6. Обучение и оценка по всем пациентам

In [ ]:
rows=[]; models={}
for pid in PATIENTS:
    torch.manual_seed(0); np.random.seed(0)
    d_tr=load_patient(f'{DATA_DIR}/{pid}-ws-training.xml')
    d_te=load_patient(f'{DATA_DIR}/{pid}-ws-testing.xml')
    xs_tr,u_tr,norm=build_features(d_tr); xs_te,_,_=build_features(d_te,norm)
    s_tr=contiguous_segments(d_tr['glucose']); s_te=contiguous_segments(d_te['glucose'])
    X0,U0,X1=make_pairs(xs_tr,u_tr,s_tr)
    m=DriftDiffNet(); train(m,X0,U0,X1,n_iter=2500)
    r=forecast_eval(m,xs_te,d_te,norm,s_te); rows.append((pid,r)); models[pid]=(m,norm,xs_te,d_te,s_te)
    print(f"{pid}: +30 RMSE {r[6]['rmse']:.2f}(p{r[6]['rmse_persist']:.2f}) "
          f"CRPS {r[6]['crps']:.2f} cov80 {r[6]['coverage80']:.2f} | "
          f"+60 RMSE {r[12]['rmse']:.2f}(p{r[12]['rmse_persist']:.2f}) "
          f"CRPS {r[12]['crps']:.2f} cov80 {r[12]['coverage80']:.2f} | "
          f"hypo AUROC {r['hypo']['auroc']:.3f}")
mean30=np.mean([r[6]['rmse'] for _,r in rows]); mean60=np.mean([r[12]['rmse'] for _,r in rows])
mcrps60=np.mean([r[12]['crps'] for _,r in rows]); mcov60=np.mean([r[12]['coverage80'] for _,r in rows])
print(f"\nmean +30 RMSE {mean30:.2f} | mean +60 RMSE {mean60:.2f} | "
      f"mean +60 CRPS {mcrps60:.2f} | mean +60 cov80 {mcov60:.2f} (цель ~0.80)")

## 7. Диагностика

In [ ]:
fig,ax=plt.subplots(1,4,figsize=(21,4.6))
pids=[p for p,_ in rows]; x=np.arange(len(pids)); w=0.2
ax[0].bar(x-1.5*w,[r[6]['rmse'] for _,r in rows],w,label='NSDE +30',color='steelblue')
ax[0].bar(x-0.5*w,[r[6]['rmse_persist'] for _,r in rows],w,label='persist +30',color='lightsteelblue')
ax[0].bar(x+0.5*w,[r[12]['rmse'] for _,r in rows],w,label='NSDE +60',color='coral')
ax[0].bar(x+1.5*w,[r[12]['rmse_persist'] for _,r in rows],w,label='persist +60',color='navajowhite')
ax[0].set_xticks(x); ax[0].set_xticklabels(pids); ax[0].set_ylabel('RMSE (mg/dL)')
ax[0].set_xlabel('patient'); ax[0].set_title('Прогноз: RMSE vs persistence')
ax[0].legend(fontsize=7); ax[0].grid(True,alpha=0.3,axis='y')

pid=pids[0]; m,norm,xs_te,d_te,s_te=models[pid]; gm,gs=norm['gm'],norm['gs']
a0,b0=max(s_te,key=lambda s:s[1]-s[0]); seg=xs_te[a0:b0]; start=200; Hh=12
cob=iob_cob_traces(d_te['carb'],TAU_CARB); iob=iob_cob_traces(d_te['insulin'],TAU_INS)
# сэмплим пучок траекторий, рисуем среднее + интервал (теперь интервал ОСМЫСЛЕННЫЙ)
with torch.no_grad():
    NS=64; X=torch.full((NS,1),float(seg[start])); c=np.full(NS,cob[a0+start]); i=np.full(NS,iob[a0+start])
    band=np.empty((NS,Hh))
    for s in range(Hh):
        c*=np.exp(-1/TAU_CARB); i*=np.exp(-1/TAU_INS)
        uu=torch.tensor(np.stack([c/norm['cs'],i/norm['is_']],1),dtype=torch.float32)
        f=m.drift(X,uu); sig=m.sigma_xu(X,uu)
        X=X+f*DT_H+sig*(DT_H**0.5)*torch.randn_like(X); band[:,s]=X.squeeze(-1).numpy()*gs+gm
mean=band.mean(0); lo=np.percentile(band,10,0); hi=np.percentile(band,90,0)
tg=np.arange(-12,Hh+1)*5
ax[1].plot(tg[:13],seg[start-12:start+1]*gs+gm,'o-',color='gray',ms=3,label='история')
ax[1].plot(tg[12:],seg[start:start+Hh+1]*gs+gm,'o-',color='steelblue',ms=3,label='факт')
ax[1].plot(tg[13:],mean,'s--',color='coral',ms=3,label='среднее NSDE')
ax[1].fill_between(tg[13:],lo,hi,color='coral',alpha=0.2,label='80% интервал')
ax[1].axvline(0,color='k',lw=0.8,ls=':'); ax[1].set_xlabel('мин от точки прогноза')
ax[1].set_ylabel('глюкоза (mg/dL)'); ax[1].set_title(f'60-мин прогноз ± интервал (pid {pid})')
ax[1].legend(fontsize=8); ax[1].grid(True,alpha=0.3)

gg=np.linspace(40,400,200); xn=torch.tensor((gg-gm)/gs,dtype=torch.float32).unsqueeze(-1)
with torch.no_grad(): fd=m.drift(xn,torch.zeros(200,2)).squeeze(-1).numpy()*gs
ax[2].plot(gg,fd,color='purple'); ax[2].axhline(0,color='k',lw=0.7)
ax[2].set_xlabel('глюкоза (mg/dL)'); ax[2].set_ylabel('дрейф dG/dt (mg/dL·ч⁻¹)')
ax[2].set_title(f'Дрейф f_θ (без еды/инсулина), pid {pid}'); ax[2].grid(True,alpha=0.3)

# НОВАЯ панель: выученная диффузия sigma(g) — растёт ли у порога гипо?
with torch.no_grad(): sd=m.sigma_xu(xn,torch.zeros(200,2)).squeeze(-1).numpy()*gs
ax[3].plot(gg,sd,color='#3fb950'); ax[3].axvline(70,color='#ff7b72',ls='--',lw=1,label='гипо 70')
ax[3].set_xlabel('глюкоза (mg/dL)'); ax[3].set_ylabel('диффузия σ (mg/dL·√ч⁻¹)')
ax[3].set_title(f'Выученная диффузия g_θ(g), pid {pid}'); ax[3].legend(fontsize=8); ax[3].grid(True,alpha=0.3)
plt.tight_layout(); plt.show()

## Итог и честные оговорки

State-dependent диффузия $g_\theta(X,u)$ оживляет распределенческую часть: теперь у модели есть CRPS, калиброванный интервал (coverage80) и P(hypo) — метрики, которые на константной $\sigma$ не имели смысла. Точечный RMSE при этом остаётся около persistence (глюкоза гладкая, точечная точность упёрта в шум сенсора) — и это **ожидаемо**: ниша Neural SDE не точка, а распределение.

**Что проверить по диагностике (§7):**
- Панель $g_\theta(g)$ должна **расти у порога гипо и/или на краях диапазона** — если σ плоская, диффузия не выучилась (подними lr головы g или убери sigma_floor-доминирование).
- coverage80 должна быть около 0.80. Сильно ниже → интервалы узкие (переуверенность); сильно выше → слишком широкие.
- CRPS NSDE против CRPS persistence-с-эмпирическим-разбросом — вот честное сравнение, а не RMSE.

**Где метод честно слаб:**
- **Автономный дрейф не предвидит необъявленные события** (болюс/еда в будущем) — потолок подхода без будущих экзогенов.
- **Одношаговое MLE ≠ оптимум для многошагового прогноза** (накопление ошибки). Апгрейд — обучение на коротких rollout'ах: это уже сделано в augmented-блоке §8.
- **Состояние 1-мерное.** Augmented Neural SDE (§8, +латент с энкодером истории) даёт память о тренде — на momentum-окнах обходит 1-D.
- COB/IOB — грубые экспоненты с фикс. $\tau$; всасывание/действие индивидуальны.

Это адаптация *той же* реализации: тот же интегратор, дрейф-MLP, Girsanov-KL — добавлена ровно state-dependent диффузия $g_\theta$ и распределенческая оценка.

## 8. Апгрейд: augmented-state Neural SDE (память о тренде)

Диагноз из §7: одномерное состояние «только текущий уровень» не несёт скорости/тренда, поэтому в momentum-окнах прогноз плоский (дрейф ≈ 0 у сет-поинта). Чиним минимально и по делу — тремя сцепленными изменениями:

1. **Augmented state** $s_t=[\,G_t,\ h_t^{(1)},\dots,h_t^{(L)}]$: к наблюдаемой глюкозе добавляем $L$ латентных измерений. Они и хранят тренд/динамический режим.
2. **Энкодер истории** $h_0=\mathrm{Enc}(G_{k-\text{CTX}:k})$ инициализирует латент из последних 60 мин — иначе модель не знает, падаем мы или растём.
3. **Multi-step rollout обучение**: латент ненаблюдаем, поэтому одношаговое псевдоправдоподобие не годится. Катим $H$ шагов с reparam-шумом ($M$ сэмплов), считаем Gaussian-NLL предсказанной глюкозы против факта + Girsanov-KL **вдоль всего пути**. Это ближе к «simulate→compare» оригинала, но условно и многошагово.

Диффузия теперь **по-компонентная** $\sigma\in\mathbb{R}^{1+L}$. Точечный прогноз = среднее по сэмплированным траекториям; разброс по ним = неопределённость (то, ради чего вообще SDE).

Это всё ещё та же реализация — Эйлер–Маруяма, MLP-дрейф, Girsanov-KL, ELBO. Прибавились только энкодер и латентные измерения.

In [ ]:
CTX, DLAT, H_TRAIN, M = 12, 4, 12, 8     # история(60м), латент, горизонт rollout(60м), MC-сэмплы

class HistEncoder(nn.Module):
    def __init__(s, ctx=CTX, dlat=DLAT, h=64):
        super().__init__(); s.net=nn.Sequential(nn.Linear(ctx,h),nn.Tanh(),nn.Linear(h,dlat))
    def forward(s,hist): return s.net(hist)

class AugDrift(nn.Module):
    def __init__(s, dlat=DLAT, d_u=2, h=64):
        super().__init__(); d=1+dlat
        s.net=nn.Sequential(nn.Linear(d+d_u,h),nn.Tanh(),nn.Linear(h,h),nn.Tanh(),nn.Linear(h,d))
        s.log_sigma=nn.Parameter(torch.full((d,),-1.5))
    def drift(s,st,u): return s.net(torch.cat([st,u],-1))
    @property
    def sigma(s): return torch.nn.functional.softplus(s.log_sigma)+1e-3

def make_windows(xs,d,norm,segs):
    cob=iob_cob_traces(d['carb'],TAU_CARB); iob=iob_cob_traces(d['insulin'],TAU_INS)
    HI,TG,C0,I0=[],[],[],[]
    for a,b in segs:
        if np.isnan(xs[a:b]).any(): continue
        for k in range(a+CTX,b-H_TRAIN):
            HI.append(xs[k-CTX:k]); TG.append(xs[k+1:k+1+H_TRAIN]); C0.append(cob[k]); I0.append(iob[k])
    t=lambda z: torch.tensor(np.array(z),dtype=torch.float32)
    return t(HI),t(TG),t(C0),t(I0)

def rollout(enc, drift, hist, c0, i0, norm, H, M, noise=True, want_kl=False):
    """вернёт предсказанную глюкозу (M,B,H); u = затухающие COB/IOB без будущих событий."""
    B=hist.shape[0]; h0=enc(hist)
    st=torch.cat([hist[:,-1:],h0],-1).unsqueeze(0).expand(M,B,-1).contiguous()
    sig=drift.sigma; sdt=DT_H**0.5
    steps=torch.arange(1,H+1,dtype=torch.float32)
    cseq=(c0[None,:]*torch.exp(-steps[:,None]/TAU_CARB))/norm['cs']
    iseq=(i0[None,:]*torch.exp(-steps[:,None]/TAU_INS))/norm['is_']
    preds=[]; kl=torch.zeros(())
    for j in range(H):
        u=torch.stack([cseq[j],iseq[j]],-1)[None].expand(M,B,2)
        f=drift.drift(st,u)
        if want_kl: kl=kl+((f**2)/(2*sig**2)).sum(-1).mean()*DT_H
        st=st+f*DT_H+(sig*sdt*torch.randn_like(st) if noise else 0.0)
        preds.append(st[...,0])
    P=torch.stack(preds,-1)
    return (P,kl) if want_kl else P

def train_aug(enc, drift, HI,TG,C0,I0, norm, n_iter=1500, bs=256, beta_kl=1e-3, lr=3e-3,
              obs_sig=0.05, log=False):
    opt=torch.optim.Adam(list(enc.parameters())+list(drift.parameters()),lr=lr)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,n_iter,1e-4); N=HI.shape[0]
    for it in range(n_iter):
        idx=torch.randint(0,N,(bs,))
        P,kl=rollout(enc,drift,HI[idx],C0[idx],I0[idx],norm,H_TRAIN,M,want_kl=True)
        mu=P.mean(0); var=P.var(0)+obs_sig**2
        nll=0.5*(((TG[idx]-mu)**2)/var+torch.log(var)).mean()
        loss=nll+beta_kl*kl
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if log and (it+1)%300==0: print(f"  it{it+1} nll={nll.item():.3f} kl={kl.item():.2f} sigma_G={drift.sigma[0].item():.3f}")
    return enc,drift

@torch.no_grad()
def eval_aug(enc, drift, xs, d, norm, segs, horizons=(6,12)):
    gm,gs=norm['gm'],norm['gs']; errs={h:[] for h in horizons}; Hmax=max(horizons)
    cob=iob_cob_traces(d['carb'],TAU_CARB); iob=iob_cob_traces(d['insulin'],TAU_INS)
    HI,C0,I0,Ks=[],[],[],[]
    for a,b in segs:
        if np.isnan(xs[a:b]).any(): continue
        for k in range(a+CTX,b-Hmax): HI.append(xs[k-CTX:k]); C0.append(cob[k]); I0.append(iob[k]); Ks.append(k)
    if not HI: return {h:dict(rmse=float('nan'),n=0) for h in horizons}
    HI=torch.tensor(np.array(HI),dtype=torch.float32); C0=torch.tensor(C0).float(); I0=torch.tensor(I0).float()
    P=rollout(enc,drift,HI,C0,I0,norm,Hmax,M=32,noise=True).mean(0)        # среднее по сэмплам
    for w,k in enumerate(Ks):
        for h in horizons:
            errs[h].append(((P[w,h-1].item()*gs+gm) - (xs[k+h]*gs+gm))**2)  # СКОБКИ обязательны!
    return {h:dict(rmse=float(np.sqrt(np.mean(errs[h]))),n=len(errs[h])) for h in horizons}

In [ ]:
# демонстрация на одном пациенте (нужны и training, и testing файлы)
PID='591'
torch.manual_seed(0); np.random.seed(0)
d_tr=load_patient(f'{DATA_DIR}/{PID}-ws-training.xml'); d_te=load_patient(f'{DATA_DIR}/{PID}-ws-testing.xml')
xs_tr,_,norm=build_features(d_tr); xs_te,_,_=build_features(d_te,norm)
s_tr=contiguous_segments(d_tr['glucose']); s_te=contiguous_segments(d_te['glucose'])

HI,TG,C0,I0=make_windows(xs_tr,d_tr,norm,s_tr)
enc=HistEncoder(); drift=AugDrift(); train_aug(enc,drift,HI,TG,C0,I0,norm,n_iter=1500,log=True)
ares=eval_aug(enc,drift,xs_te,d_te,norm,s_te)

# базовая 1-D модель для сравнения
X0,U0,X1=make_pairs(xs_tr,build_features(d_tr)[1],s_tr); base=DriftNet(); train(base,X0,U0,X1,n_iter=2500)
bres=forecast_eval(base,xs_te,d_te,norm,s_te)
print(f"\nPID {PID}  (RMSE, mg/dL)")
for h in (6,12):
    print(f"  +{h*5}min  baseline={bres[h]['rmse']:.2f}  augmented={ares[h]['rmse']:.2f}  persist={bres[h]['rmse_persist']:.2f}")

# momentum-окно: ищем самый сильный спад на тесте и рисуем baseline vs augmented
gm,gs=norm['gm'],norm['gs']; a0,b0=max(s_te,key=lambda s:s[1]-s[0]); seg=xs_te[a0:b0]
cob=iob_cob_traces(d_te['carb'],TAU_CARB); iob=iob_cob_traces(d_te['insulin'],TAU_INS)
j=max(range(CTX,len(seg)-12), key=lambda j:(seg[j]-seg[j+12])); k=a0+j
with torch.no_grad():
    hi=torch.tensor(seg[j-CTX:j],dtype=torch.float32).unsqueeze(0)
    Pa=rollout(enc,drift,hi,torch.tensor([cob[k]]).float(),torch.tensor([iob[k]]).float(),norm,12,M=64).mean(0)[0].numpy()*gs+gm
    xt=torch.tensor([[seg[j]]],dtype=torch.float32); c,i=cob[k],iob[k]; pb=[seg[j]*gs+gm]
    for st in range(12):
        c*=np.exp(-1/TAU_CARB); i*=np.exp(-1/TAU_INS)
        uu=torch.tensor([[c/norm['cs'],i/norm['is_']]],dtype=torch.float32)
        xt=xt+base.drift(xt,uu)*DT_H; pb.append(xt.item()*gs+gm)
tg=np.arange(-CTX,13)*5
plt.figure(figsize=(7,4.6))
plt.plot(tg[:CTX+1],seg[j-CTX:j+1]*gs+gm,'o-',color='gray',ms=3,label='история')
plt.plot(tg[CTX:],seg[j:j+13]*gs+gm,'o-',color='steelblue',ms=3,label='факт')
plt.plot(tg[CTX:],pb,'s--',color='coral',ms=3,label='baseline (1-D)')
plt.plot(tg[CTX+1:],Pa,'^--',color='green',ms=3,label='augmented (+latent)')
plt.axvline(0,color='k',lw=0.8,ls=':'); plt.xlabel('мин от точки прогноза'); plt.ylabel('глюкоза (mg/dL)')
plt.title(f'Momentum-окно, pid {PID}: baseline vs augmented'); plt.legend(fontsize=8); plt.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()

### Что получилось

На pid 591 augmented обходит и baseline, и persistence, причём выигрыш сосредоточен там, где и предсказывал диагноз — на длинном горизонте: **+60 мин ~34.4 vs 38.1 (baseline) vs 38.9 (persist)**; на +30 мин разница мала (память о тренде на коротком горизонте почти не нужна). На momentum-окне baseline уходит вверх/в плато, а augmented гнёт траекторию вниз вслед за фактом — латент несёт нисходящий импульс. Полный обвал он недооценивает (без знания будущего инсулина это потолок), но направление и кривизну ловит.

Дальше по убыванию отдачи: обучать на нескольких горизонтах сразу (curriculum по $H$), сделать $\sigma_\theta(s,u)$ гетероскедастичной (риск-калибровка для гипо-зоны), заменить MLP-энкодер на GRU над историей, и валидировать неопределённость (покрытие интервалов), а не только RMSE.